# 02 - EDA Business Insights (Stage 2)

This notebook generates business-oriented EDA visuals and saves all outputs to `step2_eda/plots` and `step2_eda/tables`.


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# ---------- Paths, Scaffolding, Config ----------
# Reproducible root: run notebook from project root directory.
# If needed, replace os.getcwd() with your explicit coursework folder path.
PROJECT_ROOT = os.getcwd()
DATA_PATH = os.path.join(PROJECT_ROOT, "online_shoppers_intention.csv")
TARGET = "Revenue"

STEP_DIRS = {
    "step1": os.path.join(PROJECT_ROOT, "step1_obtain_frame"),
    "step2": os.path.join(PROJECT_ROOT, "step2_eda"),
    "step3": os.path.join(PROJECT_ROOT, "step3_prepare"),
    "step4": os.path.join(PROJECT_ROOT, "step4_shortlist"),
    "step5": os.path.join(PROJECT_ROOT, "step5_finetune"),
    "step6": os.path.join(PROJECT_ROOT, "step6_present"),
}

for _, step_dir in STEP_DIRS.items():
    os.makedirs(os.path.join(step_dir, "plots"), exist_ok=True)
    os.makedirs(os.path.join(step_dir, "tables"), exist_ok=True)

STEP1_DIR = STEP_DIRS["step1"]
STEP1_PLOT_DIR = os.path.join(STEP1_DIR, "plots")
STEP1_TABLE_DIR = os.path.join(STEP1_DIR, "tables")

STEP2_DIR = STEP_DIRS["step2"]
STEP2_PLOT_DIR = os.path.join(STEP2_DIR, "plots")
STEP2_TABLE_DIR = os.path.join(STEP2_DIR, "tables")

STEP3_DIR = STEP_DIRS["step3"]
STEP3_PLOT_DIR = os.path.join(STEP3_DIR, "plots")
STEP3_TABLE_DIR = os.path.join(STEP3_DIR, "tables")

STEP4_DIR = STEP_DIRS["step4"]
STEP4_PLOT_DIR = os.path.join(STEP4_DIR, "plots")
STEP4_TABLE_DIR = os.path.join(STEP4_DIR, "tables")

STEP5_DIR = STEP_DIRS["step5"]
STEP5_PLOT_DIR = os.path.join(STEP5_DIR, "plots")
STEP5_TABLE_DIR = os.path.join(STEP5_DIR, "tables")

STEP6_DIR = STEP_DIRS["step6"]
STEP6_PLOT_DIR = os.path.join(STEP6_DIR, "plots")
STEP6_TABLE_DIR = os.path.join(STEP6_DIR, "tables")

PLOT_DPI = 300

print("Resolved PROJECT_ROOT:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Step2 plot dir:", STEP2_PLOT_DIR)
print("Step2 table dir:", STEP2_TABLE_DIR)


In [ ]:
# ---------- Helpers ----------
def load_data(path: str) -> pd.DataFrame:
    return pd.read_csv(path)


def normalize_binary_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    binary_map = {"true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0}

    for col in ["Weekend", "Revenue"]:
        if col not in out.columns:
            continue
        if pd.api.types.is_bool_dtype(out[col]):
            out[col] = out[col].astype(int)
        elif out[col].dtype == object:
            mapped = out[col].astype(str).str.strip().str.lower().map(binary_map)
            if mapped.notna().all():
                out[col] = mapped.astype(int)

    return out


def save_current_figure(filename: str, plot_dir: str = STEP2_PLOT_DIR, dpi: int = PLOT_DPI) -> str:
    path = os.path.join(plot_dir, filename)
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved plot: {path}")
    return path


def save_table(df: pd.DataFrame, filename: str, table_dir: str = STEP2_TABLE_DIR) -> str:
    path = os.path.join(table_dir, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path


def add_binomial_ci(summary_df: pd.DataFrame, success_col: str, total_col: str) -> pd.DataFrame:
    out = summary_df.copy()
    lower, upper = proportion_confint(
        count=out[success_col].values,
        nobs=out[total_col].values,
        alpha=0.05,
        method="wilson",
    )
    out["ci_lower"] = lower
    out["ci_upper"] = upper
    return out

## 1) Conversion Lift Analysis - `PageValues` Deciles


In [ ]:
def conversion_lift_pagevalues_deciles(df: pd.DataFrame, target: str = TARGET) -> pd.DataFrame:
    work = df[["PageValues", target]].dropna().copy()

    # Build quantile bins first; duplicates can reduce number of bins.
    work["pagevalues_decile"] = pd.qcut(
        work["PageValues"],
        q=10,
        duplicates="drop",
    )

    # Rename categories dynamically to D1..Dk for clean plotting.
    n_bins = len(work["pagevalues_decile"].cat.categories)
    dynamic_labels = [f"D{i}" for i in range(1, n_bins + 1)]
    work["pagevalues_decile"] = work["pagevalues_decile"].cat.rename_categories(dynamic_labels)

    summary = (
        work.groupby("pagevalues_decile", observed=True)[target]
        .agg(conversions="sum", sessions="count", conversion_rate="mean")
        .reset_index()
    )
    summary["conversion_pct"] = summary["conversion_rate"] * 100

    plt.figure(figsize=(10, 5))
    sns.barplot(data=summary, x="pagevalues_decile", y="conversion_pct", color="#2a9d8f")
    plt.title("Conversion Rate by PageValues Decile")
    plt.xlabel("PageValues Decile")
    plt.ylabel("Conversion Rate (%)")
    plt.tight_layout()
    save_current_figure("21_conversion_lift_pagevalues_deciles.png")
    plt.show()

    save_table(summary, "21_conversion_lift_pagevalues_deciles.csv")
    return summary


**Business implication**: If upper deciles show strong lift, `PageValues` is a high-signal intent feature and should be prioritized in model interpretation and thresholding strategy.


## 2) Friction Analysis - `ExitRates` vs Purchase Probability


In [ ]:
def friction_exitrates_regplot(df: pd.DataFrame, target: str = TARGET) -> pd.DataFrame:
    work = df[["ExitRates", target]].dropna().copy()

    plt.figure(figsize=(9, 5))
    sns.regplot(
        data=work,
        x="ExitRates",
        y=target,
        logistic=True,
        y_jitter=0.02,
        scatter_kws={"alpha": 0.15, "s": 12},
        line_kws={"color": "#e76f51", "linewidth": 2},
    )
    plt.title("ExitRates vs Purchase Probability (Logistic Fit)")
    plt.xlabel("ExitRates")
    plt.ylabel("Estimated Purchase Probability")
    plt.tight_layout()
    save_current_figure("22_friction_exitrates_regplot.png")
    plt.show()

    # Support table: decile-based observed conversion to complement smoothed fit
    work["exitrate_decile"] = pd.qcut(
        work["ExitRates"],
        q=10,
        duplicates="drop",
    )
    n_bins = len(work["exitrate_decile"].cat.categories)
    dynamic_labels = [f"D{i}" for i in range(1, n_bins + 1)]
    work["exitrate_decile"] = work["exitrate_decile"].cat.rename_categories(dynamic_labels)

    summary = (
        work.groupby("exitrate_decile", observed=True)[target]
        .agg(conversions="sum", sessions="count", conversion_rate="mean")
        .reset_index()
    )
    summary["conversion_pct"] = summary["conversion_rate"] * 100
    save_table(summary, "22_friction_exitrates_deciles.csv")
    return summary


**Business implication**: A downward slope indicates friction; reducing high-exit sessions may improve conversion and reduce false negatives in minority-class detection.


## 3) Segment Deep-dive - `VisitorType` and `Month` with 95% CI


In [ ]:
def segment_conversion_with_ci(df: pd.DataFrame, segment_col: str, target: str = TARGET) -> pd.DataFrame:
    work = df[[segment_col, target]].dropna().copy()

    summary = (
        work.groupby(segment_col, observed=True)[target]
        .agg(conversions="sum", sessions="count", conversion_rate="mean")
        .reset_index()
    )
    summary = add_binomial_ci(summary, success_col="conversions", total_col="sessions")
    summary["conversion_pct"] = summary["conversion_rate"] * 100
    summary["ci_lower_pct"] = summary["ci_lower"] * 100
    summary["ci_upper_pct"] = summary["ci_upper"] * 100
    return summary


def plot_segment_with_ci(summary: pd.DataFrame, segment_col: str, filename: str, title: str) -> None:
    x = np.arange(len(summary))
    y = summary["conversion_pct"].values
    yerr = np.vstack([
        y - summary["ci_lower_pct"].values,
        summary["ci_upper_pct"].values - y,
    ])

    plt.figure(figsize=(11, 5))
    plt.bar(x, y, color="#457b9d", alpha=0.9)
    plt.errorbar(x, y, yerr=yerr, fmt="none", ecolor="black", capsize=4, linewidth=1)
    plt.xticks(x, summary[segment_col].astype(str).tolist(), rotation=30, ha="right")
    plt.title(title)
    plt.xlabel(segment_col)
    plt.ylabel("Conversion Rate (%)")
    plt.tight_layout()
    save_current_figure(filename)
    plt.show()

In [ ]:
def month_sorted_summary(df: pd.DataFrame, target: str = TARGET) -> pd.DataFrame:
    month_order = ["Feb", "Mar", "Apr", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    work = df[["Month", target]].dropna().copy()
    work["Month"] = pd.Categorical(work["Month"], categories=month_order, ordered=True)

    summary = (
        work.groupby("Month", observed=True)[target]
        .agg(conversions="sum", sessions="count", conversion_rate="mean")
        .reset_index()
        .sort_values("Month")
    )
    summary = add_binomial_ci(summary, success_col="conversions", total_col="sessions")
    summary["conversion_pct"] = summary["conversion_rate"] * 100
    summary["ci_lower_pct"] = summary["ci_lower"] * 100
    summary["ci_upper_pct"] = summary["ci_upper"] * 100
    return summary

**Business implication**: Segment and seasonality effects can motivate interaction terms and time-aware validation to avoid unstable performance across months or user types.


## 4) Interaction Heatmap - `PageValues` x `ProductRelated_Duration`


In [ ]:
def interaction_heatmap_pagevalues_duration(df: pd.DataFrame, target: str = TARGET) -> pd.DataFrame:
    work = df[["PageValues", "ProductRelated_Duration", target]].dropna().copy()

    work["pv_bin"] = pd.qcut(work["PageValues"], q=6, duplicates="drop")
    work["prd_bin"] = pd.qcut(work["ProductRelated_Duration"], q=6, duplicates="drop")

    heat = (
        work.groupby(["pv_bin", "prd_bin"], observed=True)[target]
        .mean()
        .reset_index()
        .pivot(index="pv_bin", columns="prd_bin", values=target)
    )

    plt.figure(figsize=(11, 7))
    sns.heatmap(heat, annot=True, fmt=".2f", cmap="YlOrRd")
    plt.title("Purchase Probability Heatmap: PageValues x ProductRelated_Duration")
    plt.xlabel("ProductRelated_Duration (binned)")
    plt.ylabel("PageValues (binned)")
    plt.tight_layout()
    save_current_figure("24_interaction_heatmap_pagevalues_productrelated_duration.png")
    plt.show()

    save_table(heat.reset_index(), "24_interaction_heatmap_pagevalues_productrelated_duration.csv")
    return heat

**Business implication**: Strong joint lift regions indicate interaction effects that tree-based models or engineered cross-features can exploit to improve minority-class recall.


## Run Stage 2


In [ ]:
df_raw = load_data(DATA_PATH)
df = normalize_binary_columns(df_raw)

print(f"Dataset shape: {df.shape}")
display(df.head())

# 1) Conversion lift
conversion_lift_table = conversion_lift_pagevalues_deciles(df, TARGET)

# 2) Friction analysis
friction_table = friction_exitrates_regplot(df, TARGET)

# 3) Segment deep-dive
visitor_summary = segment_conversion_with_ci(df, segment_col="VisitorType", target=TARGET)
save_table(visitor_summary, "23_segment_conversion_visitor_type_ci95.csv")
plot_segment_with_ci(
    visitor_summary,
    segment_col="VisitorType",
    filename="23_segment_conversion_visitor_type_ci95.png",
    title="Conversion Rate by VisitorType (95% CI)",
)

month_summary = month_sorted_summary(df, TARGET)
save_table(month_summary, "23_segment_conversion_month_ci95.csv")
plot_segment_with_ci(
    month_summary,
    segment_col="Month",
    filename="23_segment_conversion_month_ci95.png",
    title="Monthly Conversion Rate (Chronological, 95% CI)",
)

# 4) Interaction heatmap
interaction_table = interaction_heatmap_pagevalues_duration(df, TARGET)

display(conversion_lift_table)
display(friction_table)
display(visitor_summary)
display(month_summary)
display(interaction_table)

print("All Stage 2 outputs saved to:")
print("Plots:", STEP2_PLOT_DIR)
print("Tables:", STEP2_TABLE_DIR)